# HEA Designer — RAG Pipeline (캐시 최적화 버전)

**핵심 개선:** FAISS 인덱스를 `vector_db/`에 저장해 최초 1회만 구축하고, 이후 실행은 캐시를 즉시 로드합니다.

In [1]:
# ══════════════════════════════════════════════════════════════
# 라이브러리 설치 (최초 1회만 실행)
# ══════════════════════════════════════════════════════════════

%pip install anthropic              # Claude API
%pip install faiss-cpu              # 벡터 검색 (GPU 없는 환경)
%pip install python-dotenv          # .env 파일 로드
%pip install pymupdf                # PDF 텍스트 추출 (fitz)
%pip install sentence-transformers  # 임베딩 모델
%pip install langchain-text-splitters  # PDF 청크 분할
%pip install scikit-optimize        # Bayesian Optimization (gp_minimize)
%pip install numpy pandas           # 데이터 처리
%pip install scikit-learn           # 스케일러 (모델 번들 의존)
%pip install xgboost                # ML 모델 (학습된 번들 로드용)


Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Note: you may need to restart the kernel to use updated packages.


...

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [2]:
from dotenv import load_dotenv
load_dotenv()

import os, json, re, sqlite3, pickle
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import faiss
import fitz  # pymupdf

from anthropic import Anthropic
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

# ── 경로 상수 ───────────────────────────────────────────────
BASE_DIR   = Path(r"C:/Users/admin/Desktop/alloy-designer-ai")
DB_PATH    = BASE_DIR / "data" / "hea_designer.db"
YS_MODEL_PATH = BASE_DIR / "models" / "ys_model_latest.pkl"
EL_MODEL_PATH = BASE_DIR / "models" / "el_model_latest.pkl"
PAPER_DIR  = BASE_DIR / "papers" / "processed"
VECTOR_DIR = BASE_DIR / "vector_db"
VECTOR_DIR.mkdir(exist_ok=True)

# ── FAISS 캐시 파일 경로 ────────────────────────────────────
FAISS_INDEX_PATH = VECTOR_DIR / "hea.index"
CHUNKS_PATH      = VECTOR_DIR / "chunks.json"

# ── API / 모델 상수 ─────────────────────────────────────────
CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
EMBED_MODEL  = "sentence-transformers/all-mpnet-base-v2"

client      = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
embed_model = SentenceTransformer(EMBED_MODEL)

# ── 전역 캐시 변수 (한 세션에서 1회만 로드) ────────────────
_faiss_index: Optional[faiss.Index] = None
_chunks:      Optional[List[Dict]]  = None

print("환경 로드 완료")
print(f"DB       : {DB_PATH}")
print(f"논문 수  : {len(list(PAPER_DIR.glob('*.pdf')))}")
print(f"캐시 존재: index={FAISS_INDEX_PATH.exists()}, chunks={CHUNKS_PATH.exists()}")


c:\Users\admin\Desktop\alloy-designer-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4177.26it/s]


환경 로드 완료
DB       : C:\Users\admin\Desktop\alloy-designer-ai\data\hea_designer.db
논문 수  : 37
캐시 존재: index=True, chunks=True


## 1. ML 모델 로드

In [3]:
def load_bundle(model_path: Path) -> Optional[Dict]:
    """pickle 번들 로드. 파일 없으면 None 반환."""
    if not model_path.exists():
        print(f"  ⚠ 모델 없음: {model_path}")
        return None
    with open(model_path, "rb") as f:
        bundle = pickle.load(f)
    return {"model": bundle["model"], "scaler": bundle["scaler"], "features": bundle["features"]}

ys_bundle = load_bundle(YS_MODEL_PATH)
el_bundle = load_bundle(EL_MODEL_PATH)

if ys_bundle:
    print(f"YS  feature 수: {len(ys_bundle['features'])}")
if el_bundle:
    print(f"EL  feature 수: {len(el_bundle['features'])}")


YS  feature 수: 18
EL  feature 수: 18


## 2. 질의 파싱 — `parse_user_query`

**Fix:** `system` 파라미터를 분리하고 JSON 코드블록 자동 제거 + 폴백 추가

In [4]:
# ── system 프롬프트: 영어로 작성해야 JSON 파싱 안정성 향상 ─
_PARSE_SYSTEM = """
You are an expert in BCC High-Entropy Alloy (HEA) design.
Extract the user query into the following JSON structure.
Return ONLY valid JSON — no markdown fences, no explanation.

{
  "ys_min":       <float | null>,
  "el_min":       <float | null>,
  "bcc_required": <true | false>,
  "elements":     [<str>],
  "temp_C":       <float | null>
}
"""

def parse_user_query(user_query: str) -> Dict:
    """
    자연어 질의 → 구조화된 필터 파라미터.

    Fix 목록:
    - system 파라미터 분리 (JSON 파싱 안정성 향상)
    - 코드블록(```json ... ```) 자동 제거
    - JSON 파싱 실패 시 기본값 폴백
    - Claude 원본 응답 디버그 출력
    """
    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=300,
        system=_PARSE_SYSTEM,
        messages=[{"role": "user", "content": user_query}]
    )
    text = response.content[0].text.strip()
    print(f"  [DEBUG] Claude 원본 응답: {text[:200]}")

    # ```json ... ``` 코드블록 제거
    text = re.sub(r"```(?:json)?\s*", "", text).strip().rstrip("`").strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"  ⚠ JSON 파싱 실패: {e}  원문: [{text[:100]}]")
        return {
            "ys_min": None, "el_min": None,
            "bcc_required": True, "elements": [], "temp_C": None
        }


## 3. SQLite 실험 데이터 검색 — `search_sqlite`

In [5]:
def search_sqlite(parsed_query: Dict, limit: int = 10) -> pd.DataFrame:
    """
    measurements 테이블에서 조건 검색.

    Fix 목록:
    - 컬럼명 정확히 사용: meas_YS_MPa, meas_elongation_pct
    - bcc_required=True → meas_is_BCC_single = 1
    - elements 각 원소 → alloy_{elem}_at > 0
    - ORDER BY meas_YS_MPa DESC
    - 예외 처리 + 빈 DataFrame 반환
    """
    try:
        conn = sqlite3.connect(DB_PATH)
        query = """
            SELECT
                alloy_composition_formula,
                alloy_Ti_at, alloy_Zr_at, alloy_Nb_at,
                alloy_Hf_at, alloy_Ta_at, alloy_Mo_at, alloy_W_at,
                alloy_VEC, alloy_delta_pct, alloy_Omega,
                alloy_dH_mix_kJ, alloy_dS_mix_J, alloy_Tm_mix_K,
                alloy_density_calc_gcm3, alloy_Lambda,
                alloy_phase_structure, alloy_BCC_fraction_pct,
                meas_YS_MPa, meas_UTS_MPa,
                meas_elongation_pct, meas_hardness_HV,
                meas_test_temp_C, meas_test_mode,
                meas_is_BCC_single,
                sample_process_route,
                paper_title, paper_doi, paper_year
            FROM measurements
            WHERE 1=1
        """
        params = []

        if parsed_query.get("ys_min") is not None:
            query += " AND meas_YS_MPa >= ?"
            params.append(parsed_query["ys_min"])

        if parsed_query.get("el_min") is not None:
            query += " AND meas_elongation_pct >= ?"
            params.append(parsed_query["el_min"])

        if parsed_query.get("bcc_required"):
            query += " AND meas_is_BCC_single = 1"

        if parsed_query.get("temp_C") is not None:
            query += " AND meas_test_temp_C <= ?"
            params.append(parsed_query["temp_C"] + 50)  # ±50°C 허용

        for elem in parsed_query.get("elements", []):
            col = f"alloy_{elem}_at"
            query += f" AND {col} > 0"

        query += " ORDER BY meas_YS_MPa DESC"
        query += f" LIMIT {limit}"

        df = pd.read_sql_query(query, conn, params=params)
        print(f"  → {len(df)}건 검색됨")
        conn.close()
        return df

    except Exception as e:
        print(f"  ⚠ SQLite 검색 오류: {e}")
        return pd.DataFrame()


## 4. FAISS 벡터 DB 구축 — 캐시 핵심

**`hea.index`와 `chunks.json`이 이미 있으면 즉시 로드하고, 없을 때만 PDF를 처리합니다.**

In [6]:
def _embed(texts: List[str]) -> np.ndarray:
    """배치 임베딩 → float32 ndarray."""
    return embed_model.encode(
        texts, convert_to_numpy=True, show_progress_bar=True
    ).astype("float32")


def build_faiss_index_from_pdfs() -> None:
    """
    37개 PDF → chunk → 임베딩 → FAISS 저장.
    최초 1회만 실행 필요. 이후에는 load_vector_db()가 캐시를 사용.
    """
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    all_chunks = []

    pdf_files = sorted(PAPER_DIR.glob("*.pdf"))
    print(f"PDF 수: {len(pdf_files)}")

    for pdf_path in pdf_files:
        print(f"  처리 중: {pdf_path.name}")
        try:
            doc = fitz.open(pdf_path)
            for page_idx, page in enumerate(doc):
                text = page.get_text()
                if text and len(text.strip()) > 50:
                    for chunk in splitter.split_text(text):
                        all_chunks.append({
                            "paper": pdf_path.name,
                            "page": page_idx + 1,
                            "text": chunk
                        })
        except Exception as e:
            print(f"  ⚠ 실패: {pdf_path.name} — {e}")

    print(f"총 chunk 수: {len(all_chunks)}")

    texts   = [c["text"] for c in all_chunks]
    vectors = _embed(texts)                     # ← 이 부분이 7~15분 소요

    dim   = vectors.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(vectors)

    faiss.write_index(index, str(FAISS_INDEX_PATH))
    with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=2)

    print(f"✅ FAISS 저장 완료: {FAISS_INDEX_PATH}")
    print(f"✅ Chunks 저장 완료: {CHUNKS_PATH}")


def load_vector_db():
    """
    FAISS 인덱스 + chunks 로드.
    - 파일 존재 시: 즉시 로드 (< 1초)
    - 파일 없을 시: PDF 처리 후 저장 (최초 1회)
    - 전역 캐시(_faiss_index, _chunks)로 같은 세션 내 중복 로드 방지
    """
    global _faiss_index, _chunks

    # 세션 내 이미 로드됐으면 바로 반환
    if _faiss_index is not None and _chunks is not None:
        return _faiss_index, _chunks

    # 파일이 없으면 최초 구축
    if not FAISS_INDEX_PATH.exists() or not CHUNKS_PATH.exists():
        print("⚠ 캐시 없음 → PDF 처리 시작 (최초 1회, 약 10~15분 소요)")
        build_faiss_index_from_pdfs()

    # 캐시 로드
    print("💾 FAISS 캐시 로드 중...")
    _faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))
    with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
        _chunks = json.load(f)

    print(f"✅ 로드 완료 — 벡터 수: {_faiss_index.ntotal:,}, chunk 수: {len(_chunks):,}")
    return _faiss_index, _chunks


# ── 강제 재구축이 필요할 때만 아래 주석을 해제해 실행 ──────
# build_faiss_index_from_pdfs()

# 최초 실행 시 or 캐시 없을 때 자동으로 구축
load_vector_db()


💾 FAISS 캐시 로드 중...
✅ 로드 완료 — 벡터 수: 2,570, chunk 수: 2,570


(<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x000002C9543D09F0> >,
 [{'paper': 'P001_1-s2.0-S0925838821032680-main.pdf',
   'page': 1,
   'text': 'Contents lists available at ScienceDirect \nJournal of Alloys and Compounds \njournal homepage: www.elsevier.com/locate/jalcom \nStrain rate dependent deformation behavior of BCC-structured \nTi29Zr24Nb23Hf24 high entropy alloy at elevated temperatures \nTangqing Caoa,1, Wenqi Guob,1, Wang Lua, Yunfei Xuea,⁎, Wenjun Luc, Jing Sub,  \nChristian H. Liebscherb,⁎, Chang Lib, Gerhard Dehmb \na School of Materials Science and Engineering, Beijing Institute of Technology, Beijing 100081, China \nb Max-Planck-Institut für Eisenforschung GmbH, Max-Planck-Straße 1, 40237 Düsseldorf, Germany \nc Department of Mechanical and Energy Engineering, Southern University of Science and Technology, Shenzhen 518055, China    \na r t i c l e  i n f o   \nArticle history: \nReceived 4 March 2021 \nReceived in revised

## 5. 논문 검색 — `search_papers` / `search_papers_by_sql_results`

In [7]:
def search_papers(query: str, top_k: int = 5) -> List[Dict]:
    """
    FAISS로 관련 논문 chunk 검색.
    반환: [{"paper": str, "page": int, "text": str, "score": float}, ...]
    """
    index, chunks = load_vector_db()
    qvec = _embed([query])
    distances, indices = index.search(qvec, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        if idx < 0 or idx >= len(chunks):
            continue
        chunk = chunks[idx]
        if isinstance(chunk, dict):
            results.append({
                "paper": chunk.get("paper", chunk.get("source", "unknown")),
                "page":  chunk.get("page", 0),
                "text":  chunk.get("text", chunk.get("content", str(chunk))),
                "score": float(distances[0][rank])
            })
        else:
            results.append({"paper": "unknown", "page": 0,
                             "text": str(chunk), "score": float(distances[0][rank])})
    return results


def search_papers_by_sql_results(
    sql_results: pd.DataFrame,
    user_query: str,
    top_k: int = 5
) -> List[Dict]:
    """
    SQLite 결과와 FAISS를 연동하는 핵심 함수.
    - sql_results 비어있으면 user_query만으로 검색
    - 합금명 상위 3개를 쿼리에 포함해 검색 강화
    """
    if sql_results.empty:
        print("  ℹ SQLite 결과 없음 → user_query만으로 FAISS 검색")
        return search_papers(user_query, top_k)

    # 상위 합금 조성명 추출
    formula_col = "alloy_composition_formula"
    formulas = []
    if formula_col in sql_results.columns:
        formulas = sql_results[formula_col].dropna().head(3).tolist()

    enhanced_query = user_query + " " + " ".join(formulas)
    print(f"  [강화 쿼리] {enhanced_query[:150]}")
    return search_papers(enhanced_query, top_k)


## 6. ML 예측 — `predict_properties`

In [8]:
def predict_properties(input_df: pd.DataFrame) -> Dict:
    """
    YS, 연신율 예측.
    - 모델 피처명(alloy.Ti_at 등)을 input_df 컬럼명에 자동 매핑
    - 모델 파일 없으면 None 반환
    - 예외 처리
    """
    result = {"YS_MPa": None, "EL_pct": None}

    def _align_features(df: pd.DataFrame, features: list) -> pd.DataFrame:
        """
        모델 피처명(alloy.Ti_at)과 DataFrame 컬럼명(alloy_Ti_at)이
        다를 경우 자동으로 맞춰줌.
        우선순위: 1) 정확일치 → 2) 점→언더스코어 변환 → 3) 0으로 채움
        """
        row = {}
        for feat in features:
            if feat in df.columns:
                row[feat] = df[feat].iloc[0]
            else:
                # alloy.Ti_at → alloy_Ti_at 시도
                alt = feat.replace(".", "_")
                if alt in df.columns:
                    row[feat] = df[alt].iloc[0]
                else:
                    row[feat] = 0.0
        return pd.DataFrame([row])[features]

    if ys_bundle is None:
        print("  ⚠ YS 모델 없음 → 예측 스킵")
        return result

    try:
        ys_X      = _align_features(input_df, ys_bundle["features"])
        ys_scaled = ys_bundle["scaler"].transform(ys_X)
        ys_raw    = float(ys_bundle["model"].predict(ys_scaled)[0])
        is_log    = ys_bundle.get("is_log", False)
        result["YS_MPa"] = round(
            float(__import__("numpy").expm1(ys_raw)) if is_log else ys_raw, 1
        )
    except Exception as e:
        print(f"  ⚠ YS 예측 오류: {e}")

    if el_bundle is None:
        return result

    try:
        el_X      = _align_features(input_df, el_bundle["features"])
        el_scaled = el_bundle["scaler"].transform(el_X)
        el_raw    = float(el_bundle["model"].predict(el_scaled)[0])
        is_log    = el_bundle.get("is_log", False)
        result["EL_pct"] = round(
            float(__import__("numpy").expm1(el_raw)) if is_log else el_raw, 1
        )
    except Exception as e:
        print(f"  ⚠ EL 예측 오류: {e}")

    return result


## 7. 최종 답변 생성 — `generate_final_answer`

In [9]:
_ANSWER_SYSTEM = """
You are an expert in BCC High-Entropy Alloy (HEA) design.
Answer in Korean using the data provided below.
Strictly follow this format (up to 3 recommendations):

[추천 조성 N]
- 조성:
- YS 예측/실측:
- 연신율 예측/실측:
- BCC 안정성:
- 논문 근거: (논문명, 페이지)
- 주의사항:

Rules:
- Include real experimental data source if available.
- Include paper reference for every recommendation.
- If DB data is empty, state "DB에서 조건 맞는 데이터 없음" and use only paper/ML results.
- Mark uncertain predictions with (근사값, 검증 필요).
"""

def generate_final_answer(
    user_query: str,
    parsed_query: Dict,
    sql_results: pd.DataFrame,
    paper_results: List[Dict],
    predictions: Dict
) -> str:
    """
    모든 근거를 종합해 Claude로 최종 답변 생성.
    sql_results 비어있을 때 명시적으로 알림.
    """
    # SQLite 컨텍스트
    if sql_results.empty:
        sql_ctx = "DB에서 조건 맞는 데이터 없음"
    else:
        keep_cols = ["alloy_composition_formula", "meas_YS_MPa",
                     "meas_elongation_pct", "meas_is_BCC_single",
                     "meas_BCC_fraction_pct", "paper_title"]
        keep_cols = [c for c in keep_cols if c in sql_results.columns]
        sql_ctx = sql_results[keep_cols].head(5).to_json(force_ascii=False, orient="records")

    # 논문 컨텍스트
    paper_ctx = json.dumps([
        {"paper": r["paper"], "page": r["page"], "text": r["text"][:400]}
        for r in paper_results[:5]
    ], ensure_ascii=False, indent=2)

    context = f"""
사용자 질의:
{user_query}

파싱된 조건:
{json.dumps(parsed_query, ensure_ascii=False, indent=2)}

[계층 1] 실험 DB 결과:
{sql_ctx}

[계층 2] 관련 논문 청크:
{paper_ctx}

[계층 3] ML 예측:
{json.dumps(predictions, ensure_ascii=False, indent=2)}
"""

    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=_ANSWER_SYSTEM,
        messages=[{"role": "user", "content": context}]
    )
    return response.content[0].text


## 8. 파이프라인 실행 — `main()`

In [10]:
# ══════════════════════════════════════════════════════════════
# 진단 셀 — ML 예측 디버그
# 실행: main() 호출 전에 이 셀을 먼저 실행해서 모델 상태 확인
# ══════════════════════════════════════════════════════════════

def diagnose_model():
    """
    모델 번들 내부 상태를 출력해서 예측값이 왜 이상한지 진단.
    """
    import numpy as np

    print("=" * 60)
    print("  ML 모델 진단")
    print("=" * 60)

    for name, bundle in [("YS", ys_bundle), ("EL", el_bundle)]:
        if bundle is None:
            print(f"\n[{name}] 모델 없음")
            continue

        model   = bundle["model"]
        scaler  = bundle["scaler"]
        feats   = bundle["features"]
        is_log  = bundle.get("is_log", False)

        print(f"\n[{name} 모델]")
        print(f"  피처 수    : {len(feats)}")
        print(f"  피처 목록  : {feats}")
        print(f"  is_log     : {is_log}")

        # 스케일러 중심값 확인 (학습 데이터 평균 반영)
        if hasattr(scaler, "center_"):
            centers = dict(zip(feats, scaler.center_))
        elif hasattr(scaler, "mean_"):
            centers = dict(zip(feats, scaler.mean_))
        else:
            centers = {}

        print(f"  스케일러   : {type(scaler).__name__}")
        if centers:
            print("  중심값 (상위 5개):")
            for k, v in list(centers.items())[:5]:
                print(f"    {k}: {v:.4f}")

        # Ti20Zr20Hf20Nb20Ta20 조성으로 테스트
        test_comp = {"Ti": 20, "Zr": 20, "Hf": 20, "Nb": 20, "Ta": 20}
        row = {}
        for feat in feats:
            if feat.startswith("alloy.") and feat.endswith("_at"):
                el = feat.replace("alloy.", "").replace("_at", "")
                row[feat] = test_comp.get(el, 0.0)
            elif feat == "alloy.VEC":          row[feat] = 4.23
            elif feat == "alloy.delta_pct":    row[feat] = 4.84
            elif feat == "alloy.dH_mix_kJ":    row[feat] = 2.3
            elif feat == "alloy.dS_mix_J":     row[feat] = 11.49
            elif feat == "alloy.Tm_mix_K":     row[feat] = 2307.6
            elif feat == "alloy.density_calc_gcm3": row[feat] = 8.23
            elif feat == "alloy.Omega":        row[feat] = 11.53
            elif feat == "alloy.Lambda":       row[feat] = 0.491
            elif feat == "meas.test_temp_C":   row[feat] = 25.0
            else:                              row[feat] = 0.0

        import pandas as pd
        X = pd.DataFrame([row])[feats]
        X_sc = scaler.transform(X)
        raw  = float(model.predict(X_sc)[0])
        pred = float(np.expm1(raw)) if is_log else raw

        print(f"\n  테스트 입력 (Ti20Zr20Hf20Nb20Ta20):")
        print(f"    스케일 전 주요 값: VEC={row.get('alloy.VEC','?')}, delta={row.get('alloy.delta_pct','?')}")
        print(f"    스케일 후 첫 5개: {X_sc[0][:5].round(3)}")
        print(f"    모델 원시 출력  : {raw:.4f}")
        print(f"    최종 예측값     : {pred:.1f} {'(log→expm1 역변환)' if is_log else ''}")

        # 학습 데이터 범위 힌트 (스케일러로 역추적)
        if hasattr(scaler, "center_") and hasattr(scaler, "scale_"):
            # RobustScaler: X_scaled = (X - center) / scale
            # 예측 raw를 역추적: 학습 시 타깃 범위 추정 불가이나
            # 입력 스케일이 맞는지 확인
            x_orig = X_sc[0] * scaler.scale_ + scaler.center_
            print(f"    역스케일 첫 5개 : {x_orig[:5].round(2)}")

    print("\n" + "=" * 60)
    print("  진단 완료")
    print("=" * 60)


diagnose_model()


  ML 모델 진단

[YS 모델]
  피처 수    : 18
  피처 목록  : ['alloy.Ti_at', 'alloy.Zr_at', 'alloy.Hf_at', 'alloy.V_at', 'alloy.Nb_at', 'alloy.Ta_at', 'alloy.Cr_at', 'alloy.Mo_at', 'alloy.W_at', 'alloy.Al_at', 'alloy.VEC', 'alloy.delta_pct', 'alloy.dH_mix_kJ', 'alloy.dS_mix_J', 'alloy.Tm_mix_K', 'alloy.density_calc_gcm3', 'alloy.Omega', 'alloy.Lambda']
  is_log     : False
  스케일러   : RobustScaler
  중심값 (상위 5개):
    alloy.Ti_at: 15.0000
    alloy.Zr_at: 10.0000
    alloy.Hf_at: 20.0000
    alloy.V_at: 0.0000
    alloy.Nb_at: 22.6600

  테스트 입력 (Ti20Zr20Hf20Nb20Ta20):
    스케일 전 주요 값: VEC=4.23, delta=4.84
    스케일 후 첫 5개: [ 0.196  0.4    0.     0.    -0.399]
    모델 원시 출력  : 7.4468
    최종 예측값     : 7.4 
    역스케일 첫 5개 : [20. 20. 20.  0. 20.]

[EL 모델]
  피처 수    : 18
  피처 목록  : ['alloy.Ti_at', 'alloy.Zr_at', 'alloy.Hf_at', 'alloy.V_at', 'alloy.Nb_at', 'alloy.Ta_at', 'alloy.Cr_at', 'alloy.Mo_at', 'alloy.W_at', 'alloy.Al_at', 'alloy.VEC', 'alloy.delta_pct', 'alloy.dH_mix_kJ', 'alloy.dS_mix_J', 'alloy.Tm_mix_K',

In [11]:
# ══════════════════════════════════════════════════════════════
# 공용 헬퍼 — descriptor 계산 + ML 입력 변환
# (predict_properties, BO, main 모두 이 셀에 의존)
# ══════════════════════════════════════════════════════════════

# ── 원소 물성 테이블 (근사값) ──────────────────────────────
_ELEM_PROPS = {
    "Ti": dict(VEC=4, r=1.462, Tm=1941,  rho=4.51),
    "Zr": dict(VEC=4, r=1.603, Tm=2128,  rho=6.51),
    "Hf": dict(VEC=4, r=1.580, Tm=2506,  rho=13.31),
    "V":  dict(VEC=5, r=1.316, Tm=2183,  rho=6.11),
    "Nb": dict(VEC=5, r=1.429, Tm=2750,  rho=8.57),
    "Ta": dict(VEC=5, r=1.430, Tm=3290,  rho=16.69),
    "Cr": dict(VEC=6, r=1.249, Tm=2180,  rho=7.19),
    "Mo": dict(VEC=6, r=1.363, Tm=2896,  rho=10.28),
    "W":  dict(VEC=6, r=1.371, Tm=3695,  rho=19.35),
    "Al": dict(VEC=3, r=1.432, Tm=933,   rho=2.70),
}
_ELEMENTS = ["Ti","Zr","Hf","V","Nb","Ta","Cr","Mo","W","Al"]
R_GAS = 8.314


def _calc_descriptors(comp: dict) -> dict:
    """
    {원소: at%} → descriptor dict.
    키는 모델 피처명 형식 (alloy.VEC 등).
    """
    total = sum(comp.values())
    if total <= 0:
        return {}
    c = {el: v / total for el, v in comp.items()
         if v > 0 and el in _ELEM_PROPS}
    if not c:
        return {}

    VEC    = sum(c[el] * _ELEM_PROPS[el]["VEC"] for el in c)
    r_mean = sum(c[el] * _ELEM_PROPS[el]["r"]   for el in c)
    delta  = 100 * sum(
        c[el] * (1 - _ELEM_PROPS[el]["r"] / r_mean) ** 2 for el in c
    ) ** 0.5
    dS     = -R_GAS * sum(x * np.log(x) for x in c.values())
    Tm     = sum(c[el] * _ELEM_PROPS[el]["Tm"]  for el in c)
    rho    = sum(c[el] * _ELEM_PROPS[el]["rho"] for el in c)
    Omega  = Tm * dS / (1e-3)          # dH ≈ 0 근사
    Lambda = dS / (delta ** 2 + 1e-6)

    return {
        "alloy.VEC":               round(VEC,    3),
        "alloy.delta_pct":         round(delta,  3),
        "alloy.dH_mix_kJ":         0.0,
        "alloy.dS_mix_J":          round(dS,     3),
        "alloy.Tm_mix_K":          round(Tm,     1),
        "alloy.density_calc_gcm3": round(rho,    3),
        "alloy.Omega":             round(Omega,  3),
        "alloy.Lambda":            round(Lambda, 5),
    }


def _sql_row_to_input(row: pd.Series) -> pd.DataFrame:
    """
    SQLite 결과 행 → ML 모델 입력 DataFrame.
    alloy_Ti_at (DB 컬럼) → alloy.Ti_at (모델 피처명) 자동 변환.
    DB에 descriptor가 있으면 그 값을 우선 사용.
    """
    comp = {}
    for el in _ELEMENTS:
        v = row.get(f"alloy_{el}_at") or 0
        if v > 0:
            comp[el] = float(v)

    desc = _calc_descriptors(comp)

    # DB descriptor로 덮어쓰기
    db_desc_map = {
        "alloy.VEC":               "alloy_VEC",
        "alloy.delta_pct":         "alloy_delta_pct",
        "alloy.dH_mix_kJ":         "alloy_dH_mix_kJ",
        "alloy.dS_mix_J":          "alloy_dS_mix_J",
        "alloy.Tm_mix_K":          "alloy_Tm_mix_K",
        "alloy.density_calc_gcm3": "alloy_density_calc_gcm3",
        "alloy.Omega":             "alloy_Omega",
        "alloy.Lambda":            "alloy_Lambda",
    }
    for feat_key, db_col in db_desc_map.items():
        db_val = row.get(db_col)
        if db_val is not None and not (isinstance(db_val, float) and np.isnan(db_val)):
            desc[feat_key] = float(db_val)

    comp_feats = {f"alloy.{el}_at": comp.get(el, 0.0) for el in _ELEMENTS}
    temp_C     = float(row.get("meas_test_temp_C") or 25.0)
    record     = {**comp_feats, **desc, "meas.test_temp_C": temp_C}
    return pd.DataFrame([record])


def _make_fallback_input(parsed: dict) -> pd.DataFrame:
    """
    SQLite 결과가 없을 때 parsed 원소 기반 등원자비 조성으로 입력 생성.
    """
    elements = [e for e in (parsed.get("elements") or []) if e in _ELEM_PROPS]
    if not elements:
        elements = ["Ti", "Nb", "Ta"]
    n    = len(elements)
    comp = {el: 100.0 / n for el in elements}
    desc = _calc_descriptors(comp)
    comp_feats = {f"alloy.{el}_at": comp.get(el, 0.0) for el in _ELEMENTS}
    record = {**comp_feats, **desc,
              "meas.test_temp_C": float(parsed.get("temp_C") or 25.0)}
    formula = "".join(f"{el}{round(v)}" for el, v in comp.items())
    print(f"  폴백 조성 사용: {formula}")
    return pd.DataFrame([record])


print("헬퍼 함수 로드 완료: _ELEM_PROPS, _calc_descriptors, _sql_row_to_input, _make_fallback_input")


헬퍼 함수 로드 완료: _ELEM_PROPS, _calc_descriptors, _sql_row_to_input, _make_fallback_input


In [12]:
# ══════════════════════════════════════════════════════════════
# Step 4.5 — Bayesian Optimization 기반 신규 조성 추천
#
# 흐름:
#   1. 조성 공간 정의 (각 원소 at% 범위)
#   2. 초기 랜덤 샘플 n_init개로 surrogate 초기화
#   3. Expected Improvement(EI)로 다음 후보 선택
#   4. ML 예측 → BCC 필터 → 목적함수 계산
#   5. surrogate 업데이트 반복 (n_iter 회)
#   6. 방문한 후보 중 Pareto 최적 top_n 반환
#
# 의존 패키지: scikit-optimize (pip install scikit-optimize)
# ══════════════════════════════════════════════════════════════

try:
    from skopt import gp_minimize
    from skopt.space import Real
    from skopt.utils import use_named_args
    _SKOPT_OK = True
except ImportError:
    _SKOPT_OK = False
    print("⚠ scikit-optimize 미설치 → pip install scikit-optimize")

# BCC 안정 필터 기준
BCC_RULES = {
    "VEC":       (None, 6.87),
    "delta_pct": (0.0,  8.5),
    "dH_mix_kJ": (-22.0, 7.0),
    "dS_mix_J":  (11.0, 19.5),
}

def _check_bcc(desc: dict) -> bool:
    return all([
        desc.get("alloy.VEC", 99)        < 6.87,
        0.0  <= desc.get("alloy.delta_pct", 99)  <= 8.5,
        -22.0 <= desc.get("alloy.dH_mix_kJ", 99) <= 7.0,
        11.0 <= desc.get("alloy.dS_mix_J", 0)   <= 19.5,
    ])

def _to_formula(comp: dict) -> str:
    ORDER = ["Ti","Zr","Hf","V","Nb","Ta","Cr","Mo","W","Al"]
    return "".join(f"{el}{round(comp[el])}" for el in ORDER if comp.get(el, 0) > 0)

def _objective(comp: dict, ys_min: float, el_min: float, temp_C: float) -> dict:
    """
    조성 dict → 목적함수 계산.
    반환: {"ys": float, "el": float, "score": float, "bcc": bool}
    score = -(YS * EL) : skopt는 최소화하므로 부호 반전
    BCC 통과 못하면 페널티 점수 반환.
    """
    desc = _calc_descriptors(comp)
    if not desc or not _check_bcc(desc):
        return {"ys": 0, "el": 0, "score": 9999.0, "bcc": False, "desc": desc or {}}

    record = {f"alloy.{el}_at": comp.get(el, 0.0) for el in _ELEMENTS}
    record.update(desc)
    record["meas.test_temp_C"] = temp_C
    df = pd.DataFrame([record])

    preds = predict_properties(df)
    ys = preds.get("YS_MPa") or 0.0
    el = preds.get("EL_pct") or 0.0

    # 목적함수: YS·EL 곱의 음수 (skopt 최소화 기준)
    # 목표 미달 시 페널티 추가
    penalty = 0.0
    if ys < ys_min:
        penalty += (ys_min - ys) * 10
    if el < el_min:
        penalty += (el_min - el) * 50

    score = -(ys * el) + penalty
    return {"ys": ys, "el": el, "score": score, "bcc": True, "desc": desc}


def bayesian_optimize(
    required:     List[str],
    optional:     List[str] = None,
    ys_min:       float = 800.0,
    el_min:       float = 10.0,
    temp_C:       float = 25.0,
    n_init:       int   = 20,
    n_iter:       int   = 80,
    top_n:        int   = 5,
    seed:         int   = 42,
) -> List[dict]:
    """
    Bayesian Optimization으로 신규 조성 탐색.

    Parameters
    ----------
    required  : 반드시 포함할 원소 (at% > 5 보장)
    optional  : 추가 탐색 원소 (기본: BCC 친화 원소 전체)
    ys_min    : 목표 최소 항복강도 (MPa)
    el_min    : 목표 최소 연신율 (%)
    temp_C    : 시험 온도
    n_init    : 초기 랜덤 탐색 횟수 (surrogate 초기화)
    n_iter    : BO 반복 횟수 (EI 기반 탐색)
    top_n     : 반환할 상위 조성 수

    Returns
    -------
    list of dict: [{"formula", "comp", "desc", "YS_pred", "EL_pred", "score"}, ...]
    """
    if not _SKOPT_OK:
        print("  ⚠ scikit-optimize 없음. pip install scikit-optimize 후 재실행")
        return []

    # 탐색 원소 결정
    BCC_FRIENDLY = ["Ti","Zr","Hf","V","Nb","Ta"]
    if optional is None:
        optional = [e for e in BCC_FRIENDLY if e not in required]
    search_elems = required + optional
    n_elem = len(search_elems)

    # 탐색 공간: 각 원소 at% (0~50, 이후 정규화)
    # required 원소는 최소 8 at% 보장
    spaces = []
    for el in search_elems:
        lo = 8.0 if el in required else 0.0
        spaces.append(Real(lo, 50.0, name=el))

    # 방문 기록
    visited: List[dict] = []

    def objective_fn(values):
        # values → 정규화된 조성 dict
        raw = dict(zip(search_elems, values))
        total = sum(raw.values())
        if total < 1e-6:
            return 9999.0
        comp = {el: round(v / total * 100, 1) for el, v in raw.items() if v > 0}

        result = _objective(comp, ys_min, el_min, temp_C)
        visited.append({
            "formula": _to_formula(comp),
            "comp":    comp,
            "desc":    result["desc"],
            "YS_pred": result["ys"],
            "EL_pred": result["el"],
            "bcc":     result["bcc"],
            "score":   -result["score"],   # 저장 시 부호 복원 (클수록 좋음)
        })
        return result["score"]

    print(f"  탐색 원소  : {search_elems}")
    print(f"  초기 탐색  : {n_init}회  /  BO 반복: {n_iter}회  (총 {n_init+n_iter}회)")

    result = gp_minimize(
        func        = objective_fn,
        dimensions  = spaces,
        n_calls     = n_init + n_iter,
        n_initial_points = n_init,
        acq_func    = "EI",              # Expected Improvement
        noise       = 0.01,              # 모델 예측 노이즈 추정
        random_state = seed,
        verbose     = False,
    )

    # BCC 통과 + 목표 달성한 것만 정렬
    passed = [
        r for r in visited
        if r["bcc"]
        and (r["YS_pred"] or 0) >= ys_min
        and (r["EL_pred"] or 0) >= el_min
    ]
    passed.sort(key=lambda x: x["score"], reverse=True)

    # 중복 제거 (조성식 기준)
    seen, unique = set(), []
    for r in passed:
        if r["formula"] not in seen:
            seen.add(r["formula"])
            unique.append(r)

    print(f"  총 {len(visited)}개 탐색 → BCC+목표 달성: {len(passed)}개 → 중복제거: {len(unique)}개")
    return unique[:top_n]


def search_new_compositions(
    parsed:       dict,
    n_init:       int = 20,
    n_iter:       int = 80,
    top_n:        int = 5,
) -> List[dict]:
    """
    파싱된 조건 → Bayesian Optimization → 신규 조성 상위 top_n 반환.
    main()에서 Step 4.5로 호출.
    """
    required = [e for e in (parsed.get("elements") or []) if e in _ELEM_PROPS]
    if not required:
        required = ["Ti", "Nb", "Ta"]

    ys_min = float(parsed.get("ys_min") or 800.0)
    el_min = float(parsed.get("el_min") or 10.0)
    temp_C = float(parsed.get("temp_C") or 25.0)

    print(f"  필수 원소  : {required}")
    print(f"  목표 조건  : YS ≥ {ys_min} MPa, EL ≥ {el_min} %")

    return bayesian_optimize(
        required = required,
        ys_min   = ys_min,
        el_min   = el_min,
        temp_C   = temp_C,
        n_init   = n_init,
        n_iter   = n_iter,
        top_n    = top_n,
    )


In [ ]:
def main(user_query: str = None):
    """
    5단계 RAG 파이프라인 + Step 4.5 Bayesian Optimization 신규 조성 탐색.
    """
    if user_query is None:
        user_query = """
BCC HEA 중에서
YS 1000 MPa 이상,
연신율 15% 이상 가능한 조성 추천해줘.
Ti, Nb, Ta 기반이면 좋겠어.
"""

    print("=" * 70)
    print("HEA Designer — RAG Pipeline + Bayesian Optimization")
    print("=" * 70)
    print(f"질의: {user_query.strip()}")

    # Step 1: 질의 파싱
    print("\n[1/5] 질의 파싱...")
    try:
        parsed = parse_user_query(user_query)
    except Exception as e:
        print(f"  ⚠ 파싱 실패: {e}")
        parsed = {"ys_min": None, "el_min": None, "bcc_required": True,
                  "elements": [], "temp_C": None}
    print(json.dumps(parsed, indent=2, ensure_ascii=False))

    # Step 2: SQLite 검색
    print("\n[2/5] SQLite 기존 데이터 검색...")
    try:
        sql_results = search_sqlite(parsed)
        if not sql_results.empty:
            show = [c for c in ["alloy_composition_formula","meas_YS_MPa",
                                 "meas_elongation_pct"] if c in sql_results.columns]
            print(sql_results[show].head())
    except Exception as e:
        print(f"  ⚠ SQLite 실패: {e}")
        sql_results = pd.DataFrame()

    # Step 3: 논문 검색
    print("\n[3/5] 논문 검색 (FAISS)...")
    try:
        paper_results = search_papers_by_sql_results(sql_results, user_query)
        for r in paper_results[:2]:
            print(f"  [{r['paper']}] p.{r['page']}  score={r['score']:.2f}")
            print(f"  {r['text'][:100]}...")
    except Exception as e:
        print(f"  ⚠ 논문 검색 실패: {e}")
        paper_results = []

    # Step 4: 기존 조성 ML 예측 (참고)
    print("\n[4/5] 기존 DB 상위 조성 ML 예측...")
    try:
        if not sql_results.empty:
            top_row = sql_results.iloc[0]
            print(f"  입력: {top_row.get('alloy_composition_formula', 'unknown')}")
            sample_df = _sql_row_to_input(top_row)
        else:
            sample_df = _make_fallback_input(parsed)
        ref_pred = predict_properties(sample_df)
        print(f"  YS 예측: {ref_pred['YS_MPa']} MPa  /  EL 예측: {ref_pred['EL_pct']} %")
    except Exception as e:
        print(f"  ⚠ ML 예측 실패: {e}")
        ref_pred = {"YS_MPa": None, "EL_pct": None}

    # Step 4.5: Bayesian Optimization 신규 조성 탐색
    print("\n[4.5/5] Bayesian Optimization 신규 조성 탐색...")
    try:
        new_comps = search_new_compositions(
            parsed,
            n_init = 20,   # 초기 랜덤 탐색 (늘릴수록 안정적)
            n_iter = 80,   # BO 반복 횟수 (늘릴수록 정확, 오래 걸림)
            top_n  = 5,
        )
        if new_comps:
            print("\n  ┌─ Bayesian Optimization 추천 결과 ─────────────────────────┐")
            print(f"  │ {'조성':<32} {'YS(MPa)':>8} {'EL(%)':>7} {'VEC':>5} {'δ(%)':>6} │")
            print(f"  ├{'─'*62}┤")
            for r in new_comps:
                ys  = f"{r['YS_pred']:.0f}"  if r['YS_pred'] else " N/A"
                el  = f"{r['EL_pred']:.1f}"  if r['EL_pred'] else "N/A"
                vec = f"{r['desc'].get('alloy.VEC', 0):.2f}"
                dlt = f"{r['desc'].get('alloy.delta_pct', 0):.2f}"
                print(f"  │ {r['formula']:<32} {ys:>8} {el:>7} {vec:>5} {dlt:>6} │")
            print(f"  └{'─'*62}┘")
        else:
            print("  ⚠ 조건 달성 조성 없음 — n_iter 늘리거나 목표 조건 완화 권장")
    except Exception as e:
        print(f"  ⚠ BO 탐색 실패: {e}")
        new_comps = []

    # Step 5: 최종 답변
    print("\n[5/5] 최종 답변 생성...")
    try:
        new_comp_ctx = ""
        if new_comps:
            lines = ["Bayesian Optimization 신규 탐색 조성 (ML 예측, BCC 필터 통과):"]
            for i, r in enumerate(new_comps[:3], 1):
                ys_s  = f"{r['YS_pred']:.0f} MPa" if r['YS_pred'] else "예측불가"
                el_s  = f"{r['EL_pred']:.1f} %"   if r['EL_pred'] else "예측불가"
                vec   = r['desc'].get('alloy.VEC', 0)
                delta = r['desc'].get('alloy.delta_pct', 0)
                dS    = r['desc'].get('alloy.dS_mix_J', 0)
                lines.append(
                    f"  {i}. {r['formula']}\n"
                    f"     YS={ys_s}, EL={el_s}, "
                    f"VEC={vec:.2f}, δ={delta:.2f}%, ΔSmix={dS:.2f} J/K·mol"
                )
            new_comp_ctx = "\n".join(lines)

        answer = generate_final_answer_v2(
            user_query, parsed, sql_results,
            paper_results, ref_pred, new_comp_ctx
        )
    except Exception as e:
        print(f"  ⚠ 답변 생성 실패: {e}")
        answer = "답변 생성 중 오류 발생"

    print("\n" + "=" * 70)
    print(answer)
    return answer


# ── generate_final_answer 확장 버전 ───────────────────────
_ANSWER_SYSTEM_V2 = """
You are an expert in BCC High-Entropy Alloy (HEA) design.
Answer in Korean using all provided data sources.
Strictly follow this format:

## 기존 실험 데이터 기반 추천
(SQL DB 실측값)

[추천 조성 N]
- 조성:
- YS 실측 / 연신율 실측:
- BCC 안정성:
- 논문 근거:
- 주의사항:

## Bayesian Optimization 신규 조성 추천
(미실험 조성)

[신규 조성 N]
- 조성:
- YS 예측 / 연신율 예측:
- BCC 안정성 지표: VEC, δ, ΔSmix 값 포함
- 추천 이유: 어떤 원소 조합이 왜 유리한지
- 권장 공정: 균질화 → 압연 → 어닐링 간단 제안

## 실험 우선순위
1순위 / 2순위 / 3순위 — 각 이유 포함.
"""


def generate_final_answer_v2(
    user_query: str,
    parsed_query: dict,
    sql_results: pd.DataFrame,
    paper_results: List[dict],
    ref_pred: dict,
    new_comp_ctx: str,
) -> str:
    """Bayesian Optimization 결과를 포함한 최종 답변 생성."""
    sql_ctx = (
        "DB에서 조건 맞는 데이터 없음" if sql_results.empty
        else sql_results[
            [c for c in ["alloy_composition_formula","meas_YS_MPa",
                          "meas_elongation_pct","meas_is_BCC_single",
                          "paper_title"] if c in sql_results.columns]
        ].head(5).to_json(force_ascii=False, orient="records")
    )
    paper_ctx = json.dumps([
        {"paper": r["paper"], "page": r["page"], "text": r["text"][:300]}
        for r in paper_results[:4]
    ], ensure_ascii=False, indent=2)

    context = f"""
사용자 질의: {user_query}
파싱 조건: {json.dumps(parsed_query, ensure_ascii=False)}

[계층 1] 실험 DB 결과:
{sql_ctx}

[계층 2] 관련 논문:
{paper_ctx}

[계층 3] Bayesian Optimization 신규 탐색 조성:
{new_comp_ctx if new_comp_ctx else "신규 조성 탐색 결과 없음"}

"""
    resp = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=2500,
        system=_ANSWER_SYSTEM_V2,
        messages=[{"role": "user", "content": context}]
    )
    return resp.content[0].text


# ── 실행 ──────────────────────────────────────────────────
main()


HEA Designer — RAG Pipeline + Bayesian Optimization
질의: BCC HEA 중에서
YS 1000 MPa 이상,
연신율 15% 이상 가능한 조성 추천해줘.
Ti, Nb, Ta 기반이면 좋겠어.

[1/5] 질의 파싱...
  [DEBUG] Claude 원본 응답: ```json
{
  "ys_min": 1000,
  "el_min": 15,
  "bcc_required": true,
  "elements": ["Ti", "Nb", "Ta"],
  "temp_C": null
}
```
{
  "ys_min": 1000,
  "el_min": 15,
  "bcc_required": true,
  "elements": [
    "Ti",
    "Nb",
    "Ta"
  ],
  "temp_C": null
}

[2/5] SQLite 기존 데이터 검색...
  → 5건 검색됨
  alloy_composition_formula  meas_YS_MPa  meas_elongation_pct
0      Ti19Zr19Hf19Nb19Ta19       1970.0                 28.9
1      Ti20Zr20Hf20Nb20Ta20       1600.0                 50.0
2      Ti20Zr19Hf21Nb19Ta21       1549.0                 20.8
3           Ti24V23Nb23Ta28       1391.0                 16.7
4  Ti14Zr14Nb14Ta7Mo7W7Al36       1080.0                 26.7

[3/5] 논문 검색 (FAISS)...
  [강화 쿼리] 
BCC HEA 중에서
YS 1000 MPa 이상,
연신율 15% 이상 가능한 조성 추천해줘.
Ti, Nb, Ta 기반이면 좋겠어.
 Ti19Zr19Hf19Nb19Ta19 Ti20Zr20Hf20Nb20Ta20 Ti20Zr19Hf21Nb19

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]


  [P008_2008.11671v1.pdf] p.17  score=0.76
  17 
 
Figure 4.  Theory predictions of yield strength in BCC HEAs. (a) Yield strength vs. temperatur...
  [P010_41467_2021_Article_25807.pdf] p.6  score=0.80
  pression at elevated temperatures, employing a computer-controlled Materials
Testing System (MTS) se...

[4/5] 기존 DB 상위 조성 ML 예측...
  입력: Ti19Zr19Hf19Nb19Ta19
  YS 예측: 7.6 MPa  /  EL 예측: 3.5 %

[4.5/5] Bayesian Optimization 신규 조성 탐색...
  필수 원소  : ['Ti', 'Nb', 'Ta']
  목표 조건  : YS ≥ 1000.0 MPa, EL ≥ 15.0 %
  탐색 원소  : ['Ti', 'Nb', 'Ta', 'Zr', 'Hf', 'V']
  초기 탐색  : 20회  /  BO 반복: 80회  (총 100회)
  총 100개 탐색 → BCC+목표 달성: 0개 → 중복제거: 0개
  ⚠ 조건 달성 조성 없음 — n_iter 늘리거나 목표 조건 완화 권장

[5/5] 최종 답변 생성...

# BCC HEA 조성 추천 (YS ≥1000 MPa, 연신율 ≥15%, Ti-Nb-Ta 기반)

## 기존 실험 데이터 기반 추천

### [추천 조성 1] ★최우선 추천★
- **조성**: Ti₁₉Zr₁₉Hf₁₉Nb₁₉Ta₁₉ (등원자비 5원계)
- **YS 실측**: **1970 MPa** / **연신율 실측**: **28.9%**
- **BCC 안정성**: ✓ 단상 BCC 확인
- **논문 근거**: "Microstructure and mechanical properties of in-situ nitride-reinforced

'# BCC HEA 조성 추천 (YS ≥1000 MPa, 연신율 ≥15%, Ti-Nb-Ta 기반)\n\n## 기존 실험 데이터 기반 추천\n\n### [추천 조성 1] ★최우선 추천★\n- **조성**: Ti₁₉Zr₁₉Hf₁₉Nb₁₉Ta₁₉ (등원자비 5원계)\n- **YS 실측**: **1970 MPa** / **연신율 실측**: **28.9%**\n- **BCC 안정성**: ✓ 단상 BCC 확인\n- **논문 근거**: "Microstructure and mechanical properties of in-situ nitride-reinforced refractory high-entropy alloy TiZrHfNbTa matrix composites"\n- **특징**:\n  - 요구조건 대폭 초과 달성 (YS 97% 상회, 연신율 93% 상회)\n  - Ti-Nb-Ta 기반 + Zr, Hf 첨가로 격자 뒤틀림 극대화\n  - VEC ≈ 4.4 (BCC 최적 범위)\n- **주의사항**: \n  - Hf(하프늄) 고가 원소 포함 → 원가 민감 시 조성2 고려\n  - 질화물 석출 제어 필요 (N 오염 < 0.1 wt%)\n\n---\n\n### [추천 조성 2]\n- **조성**: Ti₂₀Zr₂₀Hf₂₀Nb₂₀Ta₂₀ (등원자비, 조성1과 유사)\n- **YS 실측**: **1600 MPa** / **연신율 실측**: **50.0%**\n- **BCC 안정성**: ✓ 단상 BCC 확인\n- **논문 근거**: 동일 논문 시리즈 (TiZrHfNbTa 매트릭스 복합재)\n- **특징**:\n  - **초고연신율** 구현 (YS는 조성1보다 낮지만 여전히 요구조건 충족)\n  - 성형 가공성 우수 → 복잡 형상 부품 제조 유리\n- **주의사항**: \n  - 균질화 열처리 필수 (1200°C, 24h 권장)\n  - 급랭 시 ω상 석출 주의\n\n---\n\n### [추천 조성 3]\n- **조성**: Ti₂₀Zr₁₉Hf₂₁Nb₁₉Ta₂₁\n- **YS 실측*